In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

In [4]:
X = np.load("../src/X_data.npy")
y = np.load("../src/y_user.npy")

print(X.shape)
print(np.unique(y))

(1724, 100, 64)
[0 1 2 3 4 5 6 7]


In [5]:
unique, counts = np.unique(y, return_counts=True)

for u, c in zip(unique, counts):
    print(u, c)

0 577
1 180
2 180
3 180
4 160
5 120
6 210
7 117


In [6]:
unique, counts = np.unique(y, return_counts=True)

for u, c in zip(unique, counts):
    print(u, c)

0 577
1 180
2 180
3 180
4 160
5 120
6 210
7 117


In [7]:
selected_classes = [0, 4]

mask = np.isin(y, selected_classes)

X_binary = X[mask]
y_binary = y[mask]

print(X_binary.shape)
print(np.unique(y_binary))

(737, 100, 64)
[0 4]


In [8]:
y_binary = np.where(
    y_binary == 0,
    0,
    1
)

print(np.unique(y_binary))

[0 1]


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_binary,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print(X_train.shape)
print(X_test.shape)

(589, 100, 64)
(148, 100, 64)


In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(
    enumerate(class_weights)
)

print(class_weights)

{0: np.float64(0.6388286334056399), 1: np.float64(2.30078125)}


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)

model = Sequential([

    Conv1D(
        32,
        3,
        activation="relu",
        input_shape=(100,64)
    ),

    MaxPooling1D(2),

    Conv1D(
        64,
        3,
        activation="relu"
    ),

    MaxPooling1D(2),

    LSTM(
        64
    ),

    Dropout(0.5),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        2,
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/home/aleynagul/csi-gesture-identity-recognition/csi_env/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1780743776.849765    6658 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 49, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 47, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,554 (185.76 KB)

 Trainable params: 47,554 (185.76 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(

    X_train,
    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=32,

    class_weight=class_weights
)

Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.5202 - loss: 0.7474 - val_accuracy: 0.1949 - val_loss: 0.9554
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.5393 - loss: 0.7311 - val_accuracy: 0.5000 - val_loss: 0.6943
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.5605 - loss: 0.7162 - val_accuracy: 0.2119 - val_loss: 0.8222
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.5393 - loss: 0.6659 - val_accuracy: 0.3814 - val_loss: 0.7121
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.6369 - loss: 0.6218 - val_accuracy: 0.4492 - val_loss: 0.7394
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6943 - loss: 0.5571 - val_accuracy: 0.8051 - val_loss: 0.5084
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7558 - loss: 0.5142 - val_accuracy: 0.8729 - val_loss: 0.4256
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.7941 - loss: 0.5124 - val_accuracy: 0.8220 - v

In [13]:
model.evaluate(X_test, y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7838 - loss: 0.5335


[0.5335075855255127, 0.7837837934494019]

In [14]:
print(np.unique(y_train))
print(np.unique(y_test))

[0 1]
[0 1]


In [15]:
print(np.bincount(y_train))
print(np.bincount(y_test))

[461 128]
[116  32]


In [16]:
from sklearn.metrics import classification_report
import numpy as np

pred = model.predict(X_test)

pred_class = np.argmax(pred, axis=1)

print(
    classification_report(
        y_test,
        pred_class,
        target_names=[
            "aleyna",
            "empty"
        ]
    )
)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step
              precision    recall  f1-score   support

      aleyna       0.92      0.79      0.85       116
       empty       0.50      0.75      0.60        32

    accuracy                           0.78       148
   macro avg       0.71      0.77      0.73       148
weighted avg       0.83      0.78      0.80       148



In [17]:
import numpy as np

aleyna_idx = np.where(y_binary == 0)[0]
empty_idx = np.where(y_binary == 1)[0]

print(len(aleyna_idx))
print(len(empty_idx))

577
160


In [18]:
np.random.seed(42)

selected_aleyna = np.random.choice(
    aleyna_idx,
    size=160,
    replace=False
)

selected_empty = empty_idx

selected_idx = np.concatenate([
    selected_aleyna,
    selected_empty
])

X_balanced = X_binary[selected_idx]
y_balanced = y_binary[selected_idx]

print(X_balanced.shape)
print(np.bincount(y_balanced))

(320, 100, 64)
[160 160]


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.20,
    random_state=42,
    stratify=y_balanced
)

print(np.bincount(y_train))
print(np.bincount(y_test))

[128 128]
[32 32]


In [20]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32
)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.8284 - loss: 0.4512 - val_accuracy: 0.7885 - val_loss: 0.4315
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.7843 - loss: 0.4675 - val_accuracy: 0.7692 - val_loss: 0.4377
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.8186 - loss: 0.4173 - val_accuracy: 0.7692 - val_loss: 0.4511
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8382 - loss: 0.3704 - val_accuracy: 0.7115 - val_loss: 0.4895
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8431 - loss: 0.3410 - val_accuracy: 0.8077 - val_loss: 0.4126
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8775 - loss: 0.2847 - val_accuracy: 0.7692 - val_loss: 0.4215
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8824 - loss: 0.2912 - val_accuracy: 0.7308 - val_loss: 0.4503
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8725 - loss: 0.2789 - val_accuracy: 0.7885 - val_loss: 0.4615


In [21]:
pred = model.predict(X_test)

pred_class = np.argmax(pred, axis=1)

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred_class,
        target_names=[
            "aleyna",
            "empty"
        ]
    )
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
              precision    recall  f1-score   support

      aleyna       0.90      0.84      0.87        32
       empty       0.85      0.91      0.88        32

    accuracy                           0.88        64
   macro avg       0.88      0.88      0.87        64
weighted avg       0.88      0.88      0.87        64



In [22]:
model.save(
    "ALEYNA_EMPTY_BALANCED.h5"
)

In [24]:
sample = np.load("../realtime_sample.npy")

print("Realtime")
print("Shape:", sample.shape)
print("Min:", sample.min())
print("Max:", sample.max())
print("Mean:", sample.mean())
print("Std:", sample.std())

Realtime
Shape: (100, 64)
Min: -20.730091
Max: 4.9064546
Mean: 0.0
Std: 1.0


In [25]:
idx = np.where(y == 0)[0][0]

train_sample = X[idx]

print("Train")
print("Shape:", train_sample.shape)
print("Min:", train_sample.min())
print("Max:", train_sample.max())
print("Mean:", train_sample.mean())
print("Std:", train_sample.std())

Train
Shape: (100, 64)
Min: -96.0
Max: 122.0
Mean: 0.37119788
Std: 17.910664


In [26]:
idx = np.where(y == 0)[0][0]

train_sample = X[idx]

train_sample = (
    train_sample - np.mean(train_sample)
) / (
    np.std(train_sample) + 1e-8
)

print("Train Normalized")

print("Min:", train_sample.min())
print("Max:", train_sample.max())
print("Mean:", train_sample.mean())
print("Std:", train_sample.std())

Train Normalized
Min: -5.3806605
Max: 6.7908597
Mean: 1.1920929e-09
Std: 1.0


In [27]:
print(X.min())
print(X.max())
print(X.mean())
print(X.std())

-516.6667
339.0
0.11383226
20.004612


In [28]:
idx = np.where(y_binary == 0)[0][0]

sample = X_binary[idx]

sample_input = np.expand_dims(
    sample,
    axis=0
)

pred = model.predict(sample_input)

print(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
[[0.991091   0.00890899]]
